# Delta Lake Setup

This notebook sets up the initial Delta Lake table using the Global Superstore dataset. We'll:

1. Load the Global Superstore dataset from CSV
2. Define an appropriate schema with data types for all columns
3. Partition the data by "Ship Mode" and "Category" for performance
4. Write the data to a Delta table with optimization settings
5. Create Z-ordering on frequently queried columns
6. Set up table properties for retention and other Delta features
7. Print table metadata and sample data for verification
8. Include statistics on the initial data load

## 1. Initialize Spark Session with Delta Lake

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, to_date
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

# Set SPARK_HOME environment variable
os.environ['SPARK_HOME'] = '/usr/local/lib/python3.10/site-packages/pyspark'

# Add scripts directory to path
sys.path.append('/opt/spark/scripts')
try:
    import utils
except ImportError:
    print("Could not import utils module, creating a basic version")
    # Define a basic version of the utils module
    class Utils:
        def create_spark_session(self, app_name="Delta Lake Utils"):
            """Create a Spark session with Delta Lake configuration."""
            try:
                spark = SparkSession.builder \
                    .appName(app_name) \
                    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
                    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
                    .config("spark.databricks.delta.schema.autoMerge.enabled", "true") \
                    .config("spark.databricks.delta.optimizeWrite.enabled", "true") \
                    .config("spark.databricks.delta.autoCompact.enabled", "true") \
                    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
                    .getOrCreate()
                print(f"Created Spark session for {app_name}")
                return spark
            except Exception as e:
                print(f"Error creating Spark session with Delta Lake support: {e}")
                print("Creating basic Spark session without Delta Lake support...")
                # Create a basic Spark session without Delta Lake support
                spark = SparkSession.builder \
                    .appName(f"{app_name} (Basic)") \
                    .getOrCreate()
                print(f"Created basic Spark session for {app_name}")
                return spark
    utils = Utils()

# Create Spark session with Delta Lake support
spark = utils.create_spark_session("Delta Lake Setup")

print(f"Spark version: {spark.version}")
try:
    delta_version = spark.sql("SELECT version() as delta_version").collect()[0][0]
    print(f"Delta Lake version: {delta_version}")
except:
    print("Could not determine Delta Lake version")

## 2. Define Constants and Paths

In [ ]:
# Define paths
DATA_DIR = "/opt/spark/data"
RAW_DATA_PATH = os.path.join(DATA_DIR, "raw/Global_Superstore.csv")
DELTA_TABLE_PATH = os.path.join(DATA_DIR, "processed/global_superstore_delta")

# Create directories if they don't exist
os.makedirs(os.path.dirname(RAW_DATA_PATH), exist_ok=True)
os.makedirs(os.path.dirname(DELTA_TABLE_PATH), exist_ok=True)

print(f"Raw data path: {RAW_DATA_PATH}")
print(f"Delta table path: {DELTA_TABLE_PATH}")

## 3. Define Schema for Global Superstore Dataset

In [ ]:
# Define schema for the Global Superstore dataset
schema = StructType([
    StructField("Row ID", IntegerType(), False),
    StructField("Order ID", StringType(), False),
    StructField("Order Date", DateType(), False),
    StructField("Ship Date", DateType(), False),
    StructField("Ship Mode", StringType(), False),
    StructField("Customer ID", StringType(), False),
    StructField("Customer Name", StringType(), False),
    StructField("Segment", StringType(), False),
    StructField("Country", StringType(), False),
    StructField("City", StringType(), False),
    StructField("State", StringType(), True),
    StructField("Postal Code", StringType(), True),
    StructField("Region", StringType(), False),
    StructField("Product ID", StringType(), False),
    StructField("Category", StringType(), False),
    StructField("Sub-Category", StringType(), False),
    StructField("Product Name", StringType(), False),
    StructField("Sales", DoubleType(), False),
    StructField("Quantity", IntegerType(), False),
    StructField("Discount", DoubleType(), False),
    StructField("Profit", DoubleType(), False)
])

print(f"Schema defined with {len(schema.fields)} fields")

## 4. Load and Prepare Data

In [ ]:
# Check if the raw data file exists
if not os.path.exists(RAW_DATA_PATH):
    print(f"Raw data file not found at {RAW_DATA_PATH}")
    print("Generating sample data...")
    
    # Generate sample data using utils
    try:
        sample_data = utils.generate_test_data(num_records=10000, scenario='normal')
        # Save to CSV
        sample_data.to_csv(RAW_DATA_PATH, index=False)
        print(f"Generated {len(sample_data)} sample records and saved to {RAW_DATA_PATH}")
    except Exception as e:
        print(f"Error generating sample data: {e}")
        # Create a minimal dataset
        data = {
            'Row ID': range(1, 1001),
            'Order ID': [f"ORD-{i:05d}" for i in range(1, 1001)],
            'Order Date': ['2023-01-01'] * 1000,
            'Ship Date': ['2023-01-05'] * 1000,
            'Ship Mode': ['Standard Class'] * 250 + ['First Class'] * 250 + ['Second Class'] * 250 + ['Same Day'] * 250,
            'Customer ID': [f"CUST-{i:05d}" for i in range(1, 1001)],
            'Customer Name': [f"Customer {i}" for i in range(1, 1001)],
            'Segment': ['Consumer'] * 400 + ['Corporate'] * 300 + ['Home Office'] * 300,
            'Country': ['United States'] * 1000,
            'City': ['New York'] * 500 + ['Los Angeles'] * 500,
            'State': ['New York'] * 500 + ['California'] * 500,
            'Postal Code': ['10001'] * 500 + ['90001'] * 500,
            'Region': ['East'] * 500 + ['West'] * 500,
            'Product ID': [f"PROD-{i:05d}" for i in range(1, 1001)],
            'Category': ['Furniture'] * 300 + ['Office Supplies'] * 400 + ['Technology'] * 300,
            'Sub-Category': ['Chairs'] * 200 + ['Tables'] * 200 + ['Phones'] * 200 + ['Storage'] * 200 + ['Binders'] * 200,
            'Product Name': [f"Product {i}" for i in range(1, 1001)],
            'Sales': np.random.uniform(10, 1000, 1000).round(2),
            'Quantity': np.random.randint(1, 10, 1000),
            'Discount': np.random.choice([0, 0.1, 0.2, 0.3, 0.4, 0.5], 1000),
            'Profit': np.random.uniform(-100, 500, 1000).round(2)
        }
        pd.DataFrame(data).to_csv(RAW_DATA_PATH, index=False)
        print(f"Created 1000 minimal sample records and saved to {RAW_DATA_PATH}")

# Load data with the defined schema
try:
    # Read CSV with date format
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "false") \
        .schema(schema) \
        .option("dateFormat", "yyyy-MM-dd") \
        .load(RAW_DATA_PATH)
    
    # Convert date strings to date type if needed
    if df.schema["Order Date"].dataType.typeName() == "string":
        df = df.withColumn("Order Date", to_date(col("Order Date"), "yyyy-MM-dd"))
    if df.schema["Ship Date"].dataType.typeName() == "string":
        df = df.withColumn("Ship Date", to_date(col("Ship Date"), "yyyy-MM-dd"))
    
    # Show sample data
    print(f"Loaded {df.count()} records from {RAW_DATA_PATH}")
    df.printSchema()
    df.show(5)
except Exception as e:
    print(f"Error loading data: {e}")
    # Try with inferred schema as fallback
    print("Trying with inferred schema...")
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(RAW_DATA_PATH)
    
    # Convert date strings to date type
    df = df.withColumn("Order Date", to_date(col("Order Date"), "yyyy-MM-dd"))
    df = df.withColumn("Ship Date", to_date(col("Ship Date"), "yyyy-MM-dd"))
    
    print(f"Loaded {df.count()} records with inferred schema")
    df.printSchema()
    df.show(5)

## 5. Write to Delta Table with Partitioning

In [ ]:
# Add metadata columns
df = df.withColumn("Created_At", lit(pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")))
df = df.withColumn("Updated_At", lit(pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")))
df = df.withColumn("Source", lit("initial_load"))

# Write to Delta table with partitioning
try:
    print(f"Writing data to Delta table at {DELTA_TABLE_PATH}")
    df.write \
        .format("delta") \
        .partitionBy("Ship Mode", "Category") \
        .mode("overwrite") \
        .save(DELTA_TABLE_PATH)
    
    print("Data written successfully to Delta table")
except Exception as e:
    print(f"Error writing to Delta table: {e}")
    # Try without partitioning as fallback
    print("Trying without partitioning...")
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(DELTA_TABLE_PATH)
    print("Data written successfully to Delta table without partitioning")

## 6. Optimize and Z-Order the Delta Table

In [ ]:
# Optimize and Z-Order the Delta table
try:
    print("Optimizing Delta table...")
    spark.sql(f"OPTIMIZE delta.`{DELTA_TABLE_PATH}`").show()
    
    print("Z-Ordering Delta table by Order Date, Customer ID, and Product ID...")
    spark.sql(f"OPTIMIZE delta.`{DELTA_TABLE_PATH}` ZORDER BY (`Order Date`, `Customer ID`, `Product ID`)").show()
    
    print("Optimization and Z-Ordering completed successfully")
except Exception as e:
    print(f"Error optimizing Delta table: {e}")

## 7. Set Delta Table Properties

In [ ]:
# Set Delta table properties
try:
    print("Setting Delta table properties...")
    
    # Set retention period to 30 days
    spark.sql(f"ALTER TABLE delta.`{DELTA_TABLE_PATH}` SET TBLPROPERTIES ('delta.logRetentionDuration' = '30 days')").show()
    
    # Enable auto compaction
    spark.sql(f"ALTER TABLE delta.`{DELTA_TABLE_PATH}` SET TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')").show()
    spark.sql(f"ALTER TABLE delta.`{DELTA_TABLE_PATH}` SET TBLPROPERTIES ('delta.autoOptimize.autoCompact' = 'true')").show()
    
    # Set description
    spark.sql(f"ALTER TABLE delta.`{DELTA_TABLE_PATH}` SET TBLPROPERTIES ('comment' = 'Global Superstore dataset for Delta Lake demo')").show()
    
    print("Delta table properties set successfully")
except Exception as e:
    print(f"Error setting Delta table properties: {e}")

## 8. Verify Delta Table

In [ ]:
# Verify Delta table
try:
    print("Verifying Delta table...")
    
    # Get table details
    print("\nTable Details:")
    spark.sql(f"DESCRIBE DETAIL delta.`{DELTA_TABLE_PATH}`").show(truncate=False)
    
    # Get table history
    print("\nTable History:")
    spark.sql(f"DESCRIBE HISTORY delta.`{DELTA_TABLE_PATH}`").show(truncate=False)
    
    # Get record count
    count = spark.read.format("delta").load(DELTA_TABLE_PATH).count()
    print(f"\nTotal records: {count}")
    
    # Show sample data
    print("\nSample Data:")
    spark.read.format("delta").load(DELTA_TABLE_PATH).show(5)
    
    print("Delta table verification completed successfully")
except Exception as e:
    print(f"Error verifying Delta table: {e}")

## 9. Generate Statistics

In [ ]:
# Generate statistics
try:
    print("Generating statistics...")
    
    # Read Delta table
    delta_df = spark.read.format("delta").load(DELTA_TABLE_PATH)
    
    # Count by Ship Mode
    print("\nCount by Ship Mode:")
    delta_df.groupBy("Ship Mode").count().orderBy("count", ascending=False).show()
    
    # Count by Category
    print("\nCount by Category:")
    delta_df.groupBy("Category").count().orderBy("count", ascending=False).show()
    
    # Sales statistics
    print("\nSales Statistics:")
    delta_df.select(
        lit("Sales").alias("Metric"),
        delta_df.select("Sales").summary("min", "25%", "50%", "75%", "max").collect()[0][1].alias("Min"),
        delta_df.select("Sales").summary("min", "25%", "50%", "75%", "max").collect()[1][1].alias("25th Percentile"),
        delta_df.select("Sales").summary("min", "25%", "50%", "75%", "max").collect()[2][1].alias("Median"),
        delta_df.select("Sales").summary("min", "25%", "50%", "75%", "max").collect()[3][1].alias("75th Percentile"),
        delta_df.select("Sales").summary("min", "25%", "50%", "75%", "max").collect()[4][1].alias("Max")
    ).show()
    
    # Profit statistics
    print("\nProfit Statistics:")
    delta_df.select(
        lit("Profit").alias("Metric"),
        delta_df.select("Profit").summary("min", "25%", "50%", "75%", "max").collect()[0][1].alias("Min"),
        delta_df.select("Profit").summary("min", "25%", "50%", "75%", "max").collect()[1][1].alias("25th Percentile"),
        delta_df.select("Profit").summary("min", "25%", "50%", "75%", "max").collect()[2][1].alias("Median"),
        delta_df.select("Profit").summary("min", "25%", "50%", "75%", "max").collect()[3][1].alias("75th Percentile"),
        delta_df.select("Profit").summary("min", "25%", "50%", "75%", "max").collect()[4][1].alias("Max")
    ).show()
    
    print("Statistics generation completed successfully")
except Exception as e:
    print(f"Error generating statistics: {e}")

## 10. Setup Complete

In [ ]:
print("Delta Lake setup completed successfully!")
print(f"Delta table is available at: {DELTA_TABLE_PATH}")
print("You can now proceed with the other notebooks in the demo.")